# Notebook 2: CLIP Embedding Analysis
t-SNE/UMAP visualisation of frame embeddings; cosine similarity heatmaps; query alignment.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.manifold import TSNE
from models.clip_encoder import CLIPEncoder
from retrieval.similarity_search import SimilaritySearch
from utils.config_loader import get_config

config = get_config('../configs/config.yaml')
encoder = CLIPEncoder(config)
print('CLIP encoder ready on:', encoder.device)

In [ ]:
# --- 1. Load cached embeddings (run pipeline first) ---
# Replace with your actual cache key or load from file
import glob
npy_files = glob.glob('../.cache/*.npy')
if npy_files:
    embeddings = np.load(npy_files[0])
    print(f'Loaded embeddings: {embeddings.shape}')
else:
    print('No cache found. Run the pipeline on a video first.')
    embeddings = np.random.randn(200, 512).astype(np.float32)
    embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)

In [ ]:
# --- 2. t-SNE visualisation ---
n = min(500, len(embeddings))
subset = embeddings[:n]
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
coords = tsne.fit_transform(subset)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=np.arange(n), cmap='viridis', alpha=0.7, s=20)
plt.colorbar(scatter, label='Frame index')
plt.title(f't-SNE of {n} CLIP frame embeddings')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('tsne_embeddings.png', dpi=150)
plt.show()

In [ ]:
# --- 3. Query vs frame similarity ---
queries = [
    'person running',
    'individual loitering near entrance',
    'fight between two people',
    'car accident on road',
    'person stealing from shop',
]

search = SimilaritySearch(config)
import json, glob
idx_files = glob.glob('../.cache/*_index.json')
if idx_files:
    from data.cache_manager import CacheManager
    # Build index from cached embeddings
    from utils.types import FrameIndexEntry
    with open(idx_files[0]) as f:
        idx_data = json.load(f)
    entries = [FrameIndexEntry(**r) for r in idx_data]
    search.build_index(embeddings, entries)

    fig, axes = plt.subplots(1, len(queries), figsize=(20, 3))
    for ax, q in zip(axes, queries):
        q_vec = encoder.encode_text(q)
        results = search.search(q_vec, top_k=len(entries))
        scores = [r.cosine_score for r in results]
        ax.hist(scores, bins=30, color='steelblue', edgecolor='white')
        ax.set_title(q[:30], fontsize=8)
        ax.set_xlabel('Cosine sim')
    plt.suptitle('Score distributions per query', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No cache index found. Run pipeline first.')